# 07 · Exportación de tablas para Power BI

**Entrada:** `data/processed/` con los resultados de los notebooks 02 a 06 (`consumo_horario.parquet`, `tipos_de_dia*.parquet`, `residuos_PJME.parquet`, `metricas_ventana_movil_PJME.parquet` y `anomalias_PJME_*.parquet`)
**Salida:** `data/processed/powerbi/*.csv` (un modelo en estrella listo para importar) y `diccionario_datos.csv`

Este notebook **no calcula resultados nuevos**: reúne lo que producen los notebooks anteriores en tablas limpias, con claves coherentes, para que el panel solo tenga que importarlas y relacionarlas. El panel (`app/panel_pjm.pbix`) se construye en Power BI Desktop a partir de estas tablas.

**Decisiones de formato**

| Decisión | Motivo |
|---|---|
| CSV en UTF-8 con BOM | Power BI lo importa sin configuración y conserva tildes y eñes |
| Fechas en ISO (`2018-08-02` o `2018-08-02 07:00:00`) | Se reconocen como fecha sin ambigüedad de idioma |
| Verdadero y falso como `Sí` y `No` | Se usan directamente en segmentaciones (filtros) |
| Nombres de columna sin `%` ni espacios | Evitan tener que escribirlos entre corchetes y comillas en DAX |
| Una clave por dimensión (`region`, `fecha`, `grupo`, `estacion`) | Un solo filtro actúa sobre todas las páginas |

**Contenido**
1. Configuración y carga
2. Dimensiones
3. Tablas de hechos
4. Tablas de resultados de los notebooks 05 y 06
5. Validación del modelo
6. Exportación
7. Modelo y páginas del panel

## 1. Configuración y carga

In [1]:
%load_ext autoreload
%autoreload 2

import holidays
import numpy as np
import pandas as pd

from src.config import DESFASE_FIN_DE_HORA, HORAS_POR_DIA, PROCESSED_DIR
from src.features import ESTACIONES, agregar_variables_temporales, normalizar_por_region

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DIAS_LARGO = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
MESES_LARGO = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto',
               'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
ORDEN_ESTACION = {'Invierno': 1, 'Primavera': 2, 'Verano': 3, 'Otoño': 4}

### Parámetros de este notebook

- **`ALINEAR_HORA_CENTRAL` y `REGIONES_HORA_CENTRAL`:** deben tener **el mismo valor que en el notebook 05**. Si en el 05 se restó 1 h a `COMED` y `NI`, aquí hay que hacer lo mismo para que sus días operativos coincidan con los de `tipos_de_dia.parquet`. La sección 3 lo comprueba.
- **`REGION_PRONOSTICO`:** región de los residuos del notebook 04.
- **`SALIDA`:** carpeta de las tablas para Power BI.

In [2]:
ALINEAR_HORA_CENTRAL = False
REGIONES_HORA_CENTRAL = ['COMED', 'NI']
REGION_PRONOSTICO = 'PJME'
SALIDA = PROCESSED_DIR / 'powerbi'

requeridos = [
    'consumo_horario.parquet',
    'tipos_de_dia.parquet', 'tipos_de_dia_curvas.parquet', 'tipos_de_dia_resumen.parquet',
    'tipos_de_dia_mezcla_region.parquet', 'tipos_de_dia_silueta_k.parquet',
    f'residuos_{REGION_PRONOSTICO}.parquet', f'metricas_ventana_movil_{REGION_PRONOSTICO}.parquet',
    f'anomalias_{REGION_PRONOSTICO}_diario.parquet', f'anomalias_{REGION_PRONOSTICO}_episodios.parquet',
]
faltan = [n for n in requeridos if not (PROCESSED_DIR / n).exists()]
if faltan:
    raise FileNotFoundError(f"Faltan archivos en {PROCESSED_DIR}: {faltan}. Ejecutar antes los notebooks 02 a 06.")
print("Archivos de entrada completos.")

Archivos de entrada completos.


In [3]:
horario = pd.read_parquet(PROCESSED_DIR / 'consumo_horario.parquet')
if ALINEAR_HORA_CENTRAL:
    centrales = horario['region'].isin(REGIONES_HORA_CENTRAL)
    horario.loc[centrales, 'Datetime'] -= pd.Timedelta(hours=1)

horario = normalizar_por_region(agregar_variables_temporales(horario))
horario['tipo_calendario'] = np.select(
    [horario['es_festivo'], horario['es_fin_de_semana']], ['Festivo', 'Fin de semana'], default='Laborable'
)

tipos = pd.read_parquet(PROCESSED_DIR / 'tipos_de_dia.parquet')
print(f"{len(horario):,} filas horarias · {len(tipos):,} días con tipo de día")

1,090,343 filas horarias · 45,430 días con tipo de día


## 2. Dimensiones

Tablas pequeñas con una fila por valor de cada filtro. Son las que alimentan las segmentaciones del panel (región, fecha, estación y tipo de día).

In [4]:
# --- Región ---
rango = horario.groupby('region')['fecha'].agg(fecha_inicio='min', fecha_fin='max')
vigente = rango['fecha_fin'] >= rango['fecha_fin'].max() - pd.Timedelta(days=7)
inicio_comun = rango.loc[vigente, 'fecha_inicio'].max()
fin_comun = rango.loc[vigente, 'fecha_fin'].min()
print(f"Regiones vigentes ({int(vigente.sum())}): {list(vigente[vigente].index)}")
print(f"Periodo común (días operativos): {inicio_comun:%Y-%m-%d} → {fin_comun:%Y-%m-%d}")

dim_region = rango.assign(
    vigente=vigente,
    dias_completos=tipos.groupby('region').size(),
    MW_medio=horario.groupby('region')['MW'].mean(),
    MW_maximo=horario.groupby('region')['MW'].max(),
).reset_index()
dim_region

Regiones vigentes (10): ['AEP', 'COMED', 'DAYTON', 'DEOK', 'DOM', 'DUQ', 'EKPC', 'FE', 'PJME', 'PJMW']
Periodo común (días operativos): 2013-06-01 → 2018-08-02


,region,fecha_inicio,fecha_fin,vigente,dias_completos,MW_medio,MW_maximo
0,AEP,2004-10-01,2018-08-02,True,5054,15499.150331,25695.0
1,COMED,2011-01-01,2018-08-02,True,2771,11419.962671,23753.0
2,DAYTON,2004-10-01,2018-08-02,True,5054,2037.770495,3746.0
3,DEOK,2012-01-01,2018-08-02,True,2406,3105.063115,5445.0
4,DOM,2005-05-01,2018-08-02,True,4842,10948.863021,21651.0
5,DUQ,2005-01-01,2018-08-02,True,4962,1658.771320,3054.0
6,EKPC,2013-06-01,2018-08-02,True,1889,1464.232332,3490.0
7,FE,2011-06-01,2018-08-02,True,2619,7792.171599,14032.0
8,NI,2004-05-01,2010-12-31,False,2436,11701.084257,23631.0
9,PJME,2002-01-01,2018-08-02,True,6058,32078.927654,62009.0


In [5]:
# --- Fecha ---
anios = range(int(horario['fecha'].dt.year.min()), int(horario['fecha'].dt.year.max()) + 1)
festivos = pd.to_datetime(list(holidays.country_holidays('US', years=anios).keys()))

fechas = pd.date_range(horario['fecha'].min(), horario['fecha'].max(), freq='D')
dim_fecha = pd.DataFrame({'fecha': fechas})
dim_fecha['anio'] = dim_fecha['fecha'].dt.year
dim_fecha['trimestre'] = dim_fecha['fecha'].dt.quarter
dim_fecha['mes'] = dim_fecha['fecha'].dt.month
dim_fecha['nombre_mes'] = dim_fecha['mes'].map(dict(enumerate(MESES_LARGO, start=1)))
dim_fecha['dia_semana'] = dim_fecha['fecha'].dt.dayofweek
dim_fecha['nombre_dia'] = dim_fecha['dia_semana'].map(dict(enumerate(DIAS_LARGO)))
dim_fecha['es_fin_de_semana'] = dim_fecha['dia_semana'] >= 5
dim_fecha['es_festivo'] = dim_fecha['fecha'].isin(festivos)
dim_fecha['estacion'] = dim_fecha['mes'].map(ESTACIONES)

# --- Estación ---
dim_estacion = pd.DataFrame({'estacion': list(ORDEN_ESTACION), 'orden': list(ORDEN_ESTACION.values())})

# --- Tipo de día ---
resumen = pd.read_parquet(PROCESSED_DIR / 'tipos_de_dia_resumen.parquet')
dim_tipo_dia = resumen.rename(columns={'cumple_tamano': 'cumple_tamano_minimo'})

print(f"dim_fecha: {len(dim_fecha):,} días · dim_tipo_dia: {len(dim_tipo_dia)} grupos")
dim_tipo_dia.round(1)

dim_fecha: 7,429 días · dim_tipo_dia: 2 grupos


,grupo,n_dias,pct_dias,hora_pico,hora_minimo,amplitud,pct_fin_de_semana,pct_festivo,estacion_dominante,pct_estacion_dominante,cumple_tamano_minimo,nombre_grupo
0,1,28299,62.3,19,3,0.2,28.9,3.2,Invierno,38.1,True,G1 · Invierno · mixto
1,2,17131,37.7,17,4,0.4,28.1,2.8,Verano,63.0,True,G2 · Verano · mixto


## 3. Tablas de hechos

### 3.1 Un día por región (`hechos_diario`)

Resume cada día operativo completo de cada región: nivel, hora del pico, amplitud y tipo de día. Alimenta el calendario de colores, la línea de tiempo y los indicadores.

In [6]:
g = horario.groupby(['region', 'fecha'])
hechos_diario = g.agg(
    MW_medio=('MW', 'mean'), MW_max=('MW', 'max'), MW_min=('MW', 'min'),
    MW_rel_medio=('MW_rel', 'mean'), horas=('MW', 'size'),
).reset_index()
hechos_diario = hechos_diario[hechos_diario['horas'] == HORAS_POR_DIA].drop(columns='horas')

hora_pico = (horario.loc[g['MW'].idxmax(), ['region', 'fecha', 'hora']]
             .rename(columns={'hora': 'hora_pico'}))
hechos_diario = hechos_diario.merge(hora_pico, on=['region', 'fecha'], how='left')
hechos_diario['amplitud'] = (hechos_diario['MW_max'] - hechos_diario['MW_min']) / hechos_diario['MW_medio']

hechos_diario = (hechos_diario
                 .merge(tipos[['region', 'fecha', 'grupo', 'nombre_grupo', 'distancia_centroide']],
                        on=['region', 'fecha'], how='left')
                 .merge(dim_fecha[['fecha', 'anio', 'mes', 'estacion']], on='fecha', how='left'))

sin_grupo = int(hechos_diario['grupo'].isna().sum())
print(f"{len(hechos_diario):,} días · {sin_grupo} sin tipo de día")
if sin_grupo:
    print("Aviso: revisar que ALINEAR_HORA_CENTRAL coincida con el valor usado en el notebook 05.")
hechos_diario.head()

45,430 días · 0 sin tipo de día


,region,fecha,MW_medio,MW_max,MW_min,MW_rel_medio,hora_pico,amplitud,grupo,nombre_grupo,distancia_centroide,anio,mes,estacion
0,AEP,2004-10-01,14237.125000,15739.0,11597.0,0.918575,14,0.290930,1,G1 · Invierno · mixto,0.188548,2004,10,Otoño
1,AEP,2004-10-02,12959.250000,14082.0,11142.0,0.836126,10,0.226865,1,G1 · Invierno · mixto,0.169092,2004,10,Otoño
2,AEP,2004-10-03,12234.541667,14237.0,10430.0,0.789369,20,0.311168,1,G1 · Invierno · mixto,0.252470,2004,10,Otoño
3,AEP,2004-10-04,14352.958333,16133.0,11247.0,0.926048,20,0.340418,1,G1 · Invierno · mixto,0.233110,2004,10,Otoño
4,AEP,2004-10-05,14445.375000,16007.0,11914.0,0.932011,20,0.283343,1,G1 · Invierno · mixto,0.116946,2004,10,Otoño


### 3.2 Patrones horarios

Promedio por región, estación, día de la semana y hora (mapa de calor de 7 × 24), y por tipo de calendario (laborable, fin de semana, festivo). `MW_rel_medio` es el consumo dividido entre el promedio de su región, así que permite comparar formas entre regiones de distinto tamaño con la misma escala de color. Estas tablas ya vienen agregadas: **no responden al filtro de fecha**, solo a los de región y estación.

In [7]:
patron_hora_dia = (horario.groupby(['region', 'estacion', 'dia_semana', 'hora'])
                   .agg(MW_medio=('MW', 'mean'), MW_rel_medio=('MW_rel', 'mean'))
                   .reset_index())
patron_hora_dia['nombre_dia'] = patron_hora_dia['dia_semana'].map(dict(enumerate(DIAS_LARGO)))

patron_tipo_calendario = (horario.groupby(['region', 'estacion', 'tipo_calendario', 'hora'])
                          .agg(MW_medio=('MW', 'mean'), MW_rel_medio=('MW_rel', 'mean'))
                          .reset_index())

print(f"patron_hora_dia: {len(patron_hora_dia):,} filas · patron_tipo_calendario: {len(patron_tipo_calendario):,} filas")
patron_hora_dia.head()

patron_hora_dia: 8,064 filas · patron_tipo_calendario: 3,456 filas


,region,estacion,dia_semana,hora,MW_medio,MW_rel_medio,nombre_dia
0,AEP,Invierno,0,0,15166.917127,0.978564,Lunes
1,AEP,Invierno,0,1,14979.303867,0.966460,Lunes
2,AEP,Invierno,0,2,14943.607735,0.964157,Lunes
3,AEP,Invierno,0,3,15049.552486,0.970992,Lunes
4,AEP,Invierno,0,4,15405.441989,0.993954,Lunes


### 3.3 Tendencia y estacionalidad

Promedio por año y por mes de cada región. `completo` marca los años con casi todas las horas (los extremos de cada serie son parciales y no deben compararse con un año entero).

In [8]:
tendencia_anual = (horario.groupby(['region', 'anio'])
                   .agg(MW_medio=('MW', 'mean'), MW_rel_medio=('MW_rel', 'mean'), horas=('MW', 'size'))
                   .reset_index())
tendencia_anual['completo'] = tendencia_anual['horas'] >= 8700

estacionalidad_mensual = (horario.groupby(['region', 'mes'])
                          .agg(MW_medio=('MW', 'mean'), MW_rel_medio=('MW_rel', 'mean'))
                          .reset_index())
estacionalidad_mensual['nombre_mes'] = estacionalidad_mensual['mes'].map(dict(enumerate(MESES_LARGO, start=1)))

tendencia_anual.tail()

,region,anio,MW_medio,MW_rel_medio,horas,completo
127,PJMW,2018,5847.760222,1.043830,5136,False
128,PJM_Load,1998,28759.508712,0.966244,6600,False
129,PJM_Load,1999,29639.411758,0.995806,8760,True
130,PJM_Load,2000,30112.614469,1.011705,8784,True
131,PJM_Load,2001,30296.685959,1.017889,8760,True


### 3.4 Indicadores por región

Una fila por región para la dispersión de la página de regiones. Las columnas con sufijo `_comun` se calculan solo con las regiones vigentes en su **periodo común**, para que la comparación sea justa (las regiones cubren periodos distintos).

- `razon_inv_ver`: consumo medio de invierno entre el de verano (mayor que 1 es una región invernal).
- `amplitud_diaria`: promedio de (máximo − mínimo) del día entre el promedio del día.
- `cv_horario_pct`: desviación estándar de la demanda horaria entre su media, en %.

In [9]:
def indicadores(h, d):
    est = h.groupby(['region', 'estacion'])['MW'].mean().unstack()
    return pd.DataFrame({
        'razon_inv_ver': est['Invierno'] / est['Verano'],
        'amplitud_diaria': d.groupby('region')['amplitud'].mean(),
    })

en_comun_h = horario['region'].isin(vigente[vigente].index) & horario['fecha'].between(inicio_comun, fin_comun)
en_comun_d = hechos_diario['region'].isin(vigente[vigente].index) & hechos_diario['fecha'].between(inicio_comun, fin_comun)

indicadores_region = indicadores(horario, hechos_diario)
indicadores_region['cv_horario_pct'] = horario.groupby('region')['MW'].std() / horario.groupby('region')['MW'].mean() * 100
indicadores_region = indicadores_region.join(
    indicadores(horario[en_comun_h], hechos_diario[en_comun_d]).add_suffix('_comun')
).reset_index()
indicadores_region.round(3)

,region,razon_inv_ver,amplitud_diaria,cv_horario_pct,razon_inv_ver_comun,amplitud_diaria_comun
0,AEP,1.052,0.296,16.719,1.059,0.302
1,COMED,0.903,0.341,20.176,0.928,0.340
2,DAYTON,0.980,0.347,19.307,1.008,0.342
3,DEOK,0.932,0.345,19.318,0.945,0.347
4,DOM,0.951,0.405,22.048,0.985,0.400
5,DUQ,0.908,0.329,18.191,0.917,0.327
6,EKPC,1.196,0.440,25.874,1.196,0.440
7,FE,0.978,0.310,17.080,0.996,0.306
8,NI,0.913,0.362,20.268,NaN,NaN
9,PJME,0.928,0.373,20.151,0.950,0.369


## 4. Tablas de resultados de los notebooks 05 y 06

### 4.1 Tipos de día

Curvas medias con su banda, mezcla de tipos de día por región (historial completo y periodo común) y silueta por k.

In [10]:
curvas_tipo_dia = pd.read_parquet(PROCESSED_DIR / 'tipos_de_dia_curvas.parquet')
silueta_k = pd.read_parquet(PROCESSED_DIR / 'tipos_de_dia_silueta_k.parquet').rename(columns={'cumple_tamano': 'cumple_tamano_minimo'})

nombres = tipos.drop_duplicates('grupo').set_index('grupo')['nombre_grupo']

tipos_comun = tipos[tipos['region'].isin(vigente[vigente].index) & tipos['fecha'].between(inicio_comun, fin_comun)]
pct_total = pd.crosstab(tipos['region'], tipos['grupo'], normalize='index').mul(100).stack().rename('pct_dias_total')
pct_comun = pd.crosstab(tipos_comun['region'], tipos_comun['grupo'], normalize='index').mul(100).stack().rename('pct_dias_comun')
mezcla_region_tipo = pd.concat([pct_total, pct_comun], axis=1).reset_index()
mezcla_region_tipo['nombre_grupo'] = mezcla_region_tipo['grupo'].map(nombres)

mezcla_region_tipo.head()

,region,grupo,pct_dias_total,pct_dias_comun,nombre_grupo
0,AEP,1,67.807677,64.002118,G1 · Invierno · mixto
1,AEP,2,32.192323,35.997882,G2 · Verano · mixto
2,COMED,1,60.411404,59.343568,G1 · Invierno · mixto
3,COMED,2,39.588596,40.656432,G2 · Verano · mixto
4,DAYTON,1,65.116739,61.778719,G1 · Invierno · mixto


### 4.2 Pronóstico y error

Una fila por hora fuera de muestra (los seis pliegues de la validación en ventana móvil), con el error absoluto, el error porcentual y el tipo de día. La hora y el mes se calculan con el día operativo (`Datetime − 1 h`). El residuo es `real − predicción`: positivo significa demanda por encima de lo esperable.

In [11]:
res = pd.read_parquet(PROCESSED_DIR / f'residuos_{REGION_PRONOSTICO}.parquet')
inicio_hora = res['Datetime'] - DESFASE_FIN_DE_HORA

pronostico_horario = res.assign(
    region=REGION_PRONOSTICO,
    fecha=inicio_hora.dt.normalize(),
    hora=inicio_hora.dt.hour,
    error_abs=res['residuo'].abs(),
    error_pct=res['residuo'].abs() / res['real'] * 100,
)
pronostico_horario = (pronostico_horario
                      .merge(dim_fecha[['fecha', 'mes', 'estacion', 'es_festivo', 'es_fin_de_semana']], on='fecha', how='left')
                      .merge(tipos.loc[tipos['region'] == REGION_PRONOSTICO, ['fecha', 'grupo', 'nombre_grupo']],
                             on='fecha', how='left'))
pronostico_horario['grupo'] = pronostico_horario['grupo'].astype('Int64')

pronostico_metricas = (pd.read_parquet(PROCESSED_DIR / f'metricas_ventana_movil_{REGION_PRONOSTICO}.parquet')
                       .rename(columns={'MAPE_%': 'MAPE_pct', 'mejora_MAE_%': 'mejora_MAE_pct'})
                       .assign(region=REGION_PRONOSTICO))
pronostico_metricas['pliegue'] = pronostico_metricas['pliegue'].astype(int)

print(f"pronostico_horario: {len(pronostico_horario):,} horas · pronostico_metricas: {len(pronostico_metricas)} filas")
pronostico_metricas.round({'MAE': 2, 'RMSE': 2, 'MAPE_pct': 2, 'MASE': 2, 'mejora_MAE_pct': 2})

pronostico_horario: 52,584 horas · pronostico_metricas: 12 filas


,pliegue,modelo,MAE,RMSE,MAPE_pct,MASE,inicio_prueba,fin_prueba,mejora_MAE_pct,region
0,1,XGBoost,1551.19,2141.29,4.84,0.72,2012-08-03,2013-08-03,-24.29,PJME
1,1,Base,2048.83,2813.22,6.47,0.95,2012-08-03,2013-08-03,NaN,PJME
2,2,XGBoost,1723.19,2378.22,5.32,0.80,2013-08-03,2014-08-03,-28.26,PJME
3,2,Base,2401.88,3275.42,7.48,1.12,2013-08-03,2014-08-03,NaN,PJME
4,3,XGBoost,1688.23,2314.50,5.17,0.78,2014-08-03,2015-08-03,-25.03,PJME
5,3,Base,2251.80,3021.51,6.96,1.04,2014-08-03,2015-08-03,NaN,PJME
6,4,XGBoost,1580.75,2155.47,5.06,0.73,2015-08-03,2016-08-03,-21.49,PJME
7,4,Base,2013.47,2810.94,6.43,0.93,2015-08-03,2016-08-03,NaN,PJME
8,5,XGBoost,1727.46,2367.62,5.44,0.80,2016-08-03,2017-08-03,-20.54,PJME
9,5,Base,2174.01,2985.13,6.92,1.00,2016-08-03,2017-08-03,NaN,PJME


### 4.3 Días atípicos y episodios

Todos los días con su z modificado (para dibujar la línea de tiempo con contexto) y el contraste con episodios conocidos.

In [12]:
anomalias_dias = pd.read_parquet(PROCESSED_DIR / f'anomalias_{REGION_PRONOSTICO}_diario.parquet')
anomalias_dias = (anomalias_dias
                  .merge(tipos[['region', 'fecha', 'grupo', 'nombre_grupo', 'distancia_centroide']],
                         on=['region', 'fecha'], how='left')
                  .drop(columns=['anio', 'mes', 'estacion', 'es_festivo']))   # ya están en dim_fecha
anomalias_dias['grupo'] = anomalias_dias['grupo'].astype('Int64')

anomalias_episodios = pd.read_parquet(PROCESSED_DIR / f'anomalias_{REGION_PRONOSTICO}_episodios.parquet')
anomalias_episodios['region'] = REGION_PRONOSTICO

print(f"anomalias_dias: {len(anomalias_dias):,} días ({int(anomalias_dias['es_atipico'].sum())} atípicos) · "
      f"anomalias_episodios: {len(anomalias_episodios)} episodios")
anomalias_episodios[['episodio', 'inicio', 'fin', 'estado', 'n_dias_atipicos', 'verificado']]

anomalias_dias: 2,191 días (22 atípicos) · anomalias_episodios: 10 episodios


,episodio,inicio,fin,estado,n_dias_atipicos,verificado
0,Ola de calor de agosto de 2006 (récord de PJM ...,2006-08-01,2006-08-03,fuera de rango,0,True
1,Ola de calor de julio de 2011,2011-07-20,2011-07-22,fuera de rango,0,True
2,Huracán Sandy,2012-10-29,2012-10-31,coincide,1,True
3,Ola de calor de julio de 2013 (pico de PJM el ...,2013-07-15,2013-07-19,sin coincidencia,0,True
4,Calor de septiembre de 2013,2013-09-10,2013-09-12,coincide,2,True
5,Vórtice polar de enero de 2014 (récord inverna...,2014-01-06,2014-01-08,coincide,1,True
6,Vórtice polar de fines de enero de 2014,2014-01-23,2014-01-29,coincide,1,True
7,Ola de frío de febrero de 2015 (récord inverna...,2015-02-14,2015-02-21,coincide,1,True
8,Tormenta Jonas,2016-01-22,2016-01-24,sin coincidencia,0,False
9,Ola de frío de enero de 2018,2018-01-01,2018-01-07,sin coincidencia,0,True


## 5. Validación del modelo

Antes de exportar se comprueba que las tablas se pueden relacionar: las claves de cada tabla de hechos deben existir en su dimensión, y las claves de las dimensiones deben ser únicas. Una clave huérfana aparecería en Power BI como una fila en blanco en los filtros.

In [13]:
tablas = {
    'dim_region': dim_region, 'dim_fecha': dim_fecha, 'dim_estacion': dim_estacion, 'dim_tipo_dia': dim_tipo_dia,
    'hechos_diario': hechos_diario,
    'patron_hora_dia': patron_hora_dia, 'patron_tipo_calendario': patron_tipo_calendario,
    'tendencia_anual': tendencia_anual, 'estacionalidad_mensual': estacionalidad_mensual,
    'indicadores_region': indicadores_region,
    'curvas_tipo_dia': curvas_tipo_dia, 'mezcla_region_tipo': mezcla_region_tipo, 'silueta_k': silueta_k,
    'pronostico_horario': pronostico_horario, 'pronostico_metricas': pronostico_metricas,
    'anomalias_dias': anomalias_dias, 'anomalias_episodios': anomalias_episodios,
}

dimensiones = {'region': ('dim_region', 'region'), 'fecha': ('dim_fecha', 'fecha'),
               'estacion': ('dim_estacion', 'estacion'), 'grupo': ('dim_tipo_dia', 'grupo')}

# 1. Las claves de las dimensiones son únicas
for clave, (dim, col) in dimensiones.items():
    assert tablas[dim][col].is_unique, f"{dim}[{col}] tiene valores repetidos"

# 2. Toda clave de una tabla de hechos existe en su dimensión
filas = []
for nombre, t in tablas.items():
    if nombre.startswith('dim_'):
        continue
    for clave, (dim, col) in dimensiones.items():
        if clave in t.columns:
            valores = t[clave].dropna()
            huerfanos = int((~valores.isin(tablas[dim][col])).sum())
            filas.append({'tabla': nombre, 'clave': clave, 'dimension': dim, 'huerfanos': huerfanos})
comprobacion = pd.DataFrame(filas)
display(comprobacion)
assert comprobacion['huerfanos'].eq(0).all(), "Hay claves sin su dimensión: revisar las tablas con huérfanos"
print("Modelo coherente: todas las claves tienen su dimensión.")

,tabla,clave,dimension,huerfanos
0,hechos_diario,region,dim_region,0
1,hechos_diario,fecha,dim_fecha,0
2,hechos_diario,estacion,dim_estacion,0
3,hechos_diario,grupo,dim_tipo_dia,0
4,patron_hora_dia,region,dim_region,0
5,patron_hora_dia,estacion,dim_estacion,0
6,patron_tipo_calendario,region,dim_region,0
7,patron_tipo_calendario,estacion,dim_estacion,0
8,tendencia_anual,region,dim_region,0
9,estacionalidad_mensual,region,dim_region,0


Modelo coherente: todas las claves tienen su dimensión.


In [14]:
# 3. Cobertura de los filtros del panel: región, fecha y estación
print("Tablas con filtro de región:", [n for n, t in tablas.items() if 'region' in t.columns and not n.startswith('dim_')])
print("Tablas con filtro de fecha:", [n for n, t in tablas.items() if 'fecha' in t.columns and not n.startswith('dim_')])
print("Tablas con filtro de estación:", [n for n, t in tablas.items() if 'estacion' in t.columns and not n.startswith('dim_')])

Tablas con filtro de región: ['hechos_diario', 'patron_hora_dia', 'patron_tipo_calendario', 'tendencia_anual', 'estacionalidad_mensual', 'indicadores_region', 'mezcla_region_tipo', 'pronostico_horario', 'pronostico_metricas', 'anomalias_dias', 'anomalias_episodios']
Tablas con filtro de fecha: ['hechos_diario', 'pronostico_horario', 'anomalias_dias']
Tablas con filtro de estación: ['hechos_diario', 'patron_hora_dia', 'patron_tipo_calendario', 'pronostico_horario']


## 6. Exportación

Cada tabla se escribe como CSV en `data/processed/powerbi/`. Los verdaderos y falsos pasan a `Sí` y `No`, las fechas a texto ISO (con la hora solo si la tabla la tiene) y los decimales se redondean a 4 cifras. Se escribe además `diccionario_datos.csv` con las columnas y tipos de cada tabla.

In [15]:
def preparar(tabla):
    t = tabla.copy()
    for c in t.columns:
        if pd.api.types.is_bool_dtype(t[c]):
            t[c] = t[c].map({True: 'Sí', False: 'No'})
        elif pd.api.types.is_datetime64_any_dtype(t[c]):
            con_hora = (t[c].dropna() != t[c].dropna().dt.normalize()).any()
            t[c] = t[c].dt.strftime('%Y-%m-%d %H:%M:%S' if con_hora else '%Y-%m-%d')
        elif pd.api.types.is_float_dtype(t[c]):
            t[c] = t[c].round(4)
    return t

SALIDA.mkdir(parents=True, exist_ok=True)
resumen_export = []
for nombre, tabla in tablas.items():
    ruta = SALIDA / f'{nombre}.csv'
    preparar(tabla).to_csv(ruta, index=False, encoding='utf-8-sig')
    resumen_export.append({'tabla': nombre, 'filas': len(tabla), 'columnas': tabla.shape[1],
                           'tamano_MB': round(ruta.stat().st_size / 1e6, 2)})
resumen_export = pd.DataFrame(resumen_export)
print(f"{len(resumen_export)} tablas en {SALIDA}")
resumen_export

17 tablas en C:\Users\lukas\Desktop\Projects\energy-consumption-analysis\data\processed\powerbi


,tabla,filas,columnas,tamano_MB
0,dim_region,12,7,0.00
1,dim_fecha,7429,10,0.39
2,dim_estacion,4,2,0.00
3,dim_tipo_dia,2,12,0.00
4,hechos_diario,45430,14,4.81
5,patron_hora_dia,8064,7,0.35
6,patron_tipo_calendario,3456,6,0.15
7,tendencia_anual,132,6,0.00
8,estacionalidad_mensual,144,5,0.00
9,indicadores_region,12,6,0.00


In [16]:
diccionario = pd.DataFrame([
    {'tabla': nombre, 'columna': c, 'tipo': str(t[c].dtype), 'ejemplo': t[c].dropna().iloc[0] if t[c].notna().any() else ''}
    for nombre, t in tablas.items() for c in t.columns
])
diccionario.to_csv(SALIDA / 'diccionario_datos.csv', index=False, encoding='utf-8-sig')
print(f"diccionario_datos.csv: {len(diccionario)} columnas documentadas")
diccionario.head(10)

diccionario_datos.csv: 146 columnas documentadas


,tabla,columna,tipo,ejemplo
0,dim_region,region,str,AEP
1,dim_region,fecha_inicio,datetime64[us],2004-10-01 00:00:00
2,dim_region,fecha_fin,datetime64[us],2018-08-02 00:00:00
3,dim_region,vigente,bool,True
4,dim_region,dias_completos,int64,5054
5,dim_region,MW_medio,float64,15499.150331
6,dim_region,MW_maximo,float64,25695.0
7,dim_fecha,fecha,datetime64[us],1998-04-01 00:00:00
8,dim_fecha,anio,int32,1998
9,dim_fecha,trimestre,int32,2


## 7. Modelo y páginas del panel

### Modelo de datos (estrella)

Importar los CSV de `data/processed/powerbi/` en Power BI Desktop y crear estas relaciones (uno a varios, filtro en un solo sentido, de la dimensión a la tabla de hechos):

| Dimensión | Clave | Tablas relacionadas |
|---|---|---|
| `dim_region` | `region` | `hechos_diario`, `patron_hora_dia`, `patron_tipo_calendario`, `tendencia_anual`, `estacionalidad_mensual`, `indicadores_region`, `mezcla_region_tipo`, `pronostico_horario`, `pronostico_metricas`, `anomalias_dias`, `anomalias_episodios` |
| `dim_fecha` | `fecha` | `hechos_diario`, `pronostico_horario`, `anomalias_dias` |
| `dim_estacion` | `estacion` | `dim_fecha` (la estación de cada fecha), `patron_hora_dia`, `patron_tipo_calendario` |
| `dim_tipo_dia` | `grupo` | `hechos_diario`, `curvas_tipo_dia`, `mezcla_region_tipo`, `pronostico_horario`, `anomalias_dias` |

Ordenar `nombre_mes` por `mes`, `nombre_dia` por `dia_semana` y `estacion` por `orden`; si no, las segmentaciones salen en orden alfabético.

**Filtros del panel (criterio de cierre).** Con una segmentación sincronizada de región, fecha y estación en todas las páginas: la **región** y la **estación** actúan sobre todas las tablas. La **fecha** actúa sobre `hechos_diario`, `pronostico_horario` y `anomalias_dias`, pero no sobre las tablas ya agregadas (`patron_*`, `tendencia_anual`, `estacionalidad_mensual`, `indicadores_region`, `mezcla_region_tipo`, `curvas_tipo_dia`). Conviene indicarlo en una nota del panel.

### Páginas, preguntas y gráficos

Cada vista se justifica por una tarea y un canal perceptual (ver `docs/03_metodos_y_herramientas.md`).

| Página | Pregunta | Tablas | Visuales (canal) |
|---|---|---|---|
| Resumen | Panorama | `dim_region`, `hechos_diario`, `silueta_k`, `pronostico_metricas`, `anomalias_dias` | Tarjetas (días, regiones, silueta, MAE y MASE, días atípicos); líneas de la tendencia anual (posición sobre escala común) |
| Patrones | P1, P2 | `patron_hora_dia`, `patron_tipo_calendario`, `estacionalidad_mensual`, `tendencia_anual` | Mapa de calor 7 × 24 con la misma escala en todas las regiones (color secuencial de un matiz); líneas laborable, fin de semana y festivo (posición) |
| Tipos de día | P1 | `curvas_tipo_dia`, `hechos_diario`, `dim_tipo_dia` | Líneas con banda de dispersión (posición); calendario con color categórico (k ≤ 8); barras con el % de días de cada tipo (longitud) |
| Regiones | P2 | `mezcla_region_tipo`, `indicadores_region` | Barras apiladas al 100 % (longitud sobre línea base común); dispersión con etiquetas (posición en dos ejes) |
| Anomalías | P3 | `anomalias_dias`, `anomalias_episodios` | Línea de tiempo con puntos resaltados (pocos puntos destacan sobre el resto); tabla de episodios con su estado y fuente |
| Pronóstico | P4 | `pronostico_horario`, `pronostico_metricas` | Barras del error por hora, mes y tipo de día (longitud); líneas del MAE por pliegue (posición) |

### Medidas(DAX)

```dax
Dias = COUNTROWS ( hechos_diario )
MW medio = AVERAGE ( hechos_diario[MW_medio] )
Dias atipicos = CALCULATE ( COUNTROWS ( anomalias_dias ), anomalias_dias[es_atipico] = "Sí" )
MAE = AVERAGE ( pronostico_horario[error_abs] )
MAPE = AVERAGE ( pronostico_horario[error_pct] )
```
